# 🎯 [MOV02_M04] 적 위협 반경 회피 및 자산 적재 효율을 고려한 최적 수송 경로 추천 모델 고도화
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m04_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M04 |
| **미션 주제** | 적 위협 반경 회피 및 자산 적재 효율을 고려한 최적 수송 경로 추천 모델 고도화 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. 적 위협 반경 회피 및 자산 적재 효율을 고려한 최적 수송 경로 추천 모델 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

적 위협 반경 회피 및 적재 효율을 고려한 최적 수송 경로 추천 모델을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **적 위협 반경 회피와 적재 효율(차종 조합)을 함께 고려한 최적 수송 경로 추천 모델로 고도화한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **간선 비용에 소요시간과 위협 위험 가중치를 함께 넣고, 과거 수송통제관 승인 사례(가상)로 위험 가중치를 학습 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ai/mov02_intelligence_module.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추천 사유의 합리성 및 실무 수용성

#### 미션 PASS 조건

- 정량 기준: 추천/분류 정확도 85% 이상 달성


- 진행 기준: AI 추론 모듈 구현 및 벤치마크 테스트 완료


- 검증 방법: 과거 승인/처리 완료 데이터와의 추천 일치도 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 위협 구역 회피 비용과 적재 효율(차종 조합)을 함께 고려하는 수송 경로 추천 모델을 구현한다. 과거 수송통제관 승인 사례로 위험 가중치를 학습해 추천 일치도(Top-1·Top-3)를 검증하고, 추천 사유와 유사도 점수를 제시한다."""
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    # [TODO : 학생 실습 4] 적 위협 반경 회피 및 자산 적재 효율을 고려한 최적 수송 경로 추천 모델 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def edge_minutes(e, asset="truck"):
    # [TODO : 학생 실습 4] 적 위협 반경 회피 및 자산 적재 효율을 고려한 최적 수송 경로 추천 모델 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

nodes, edges = make_network()
DEPOT, UNITS = "N0000", [f"N{r:02d}07" for r in range(6)]
for e in edges:
    if e["id"] in ("E017", "E040", "E046", "E071"):
        e["open"] = False
LAMS = [0.0, 0.3, 0.7, 1.5, 3.0]           # 위험 회피 강도별 후보 경로 생성


def candidates(dest, threats, kind="truck", gross=15, k=5):
    """위험 회피 강도(lam)를 바꾸고, 이미 찾은 경로 구간에 벌점을 주어 서로 다른 후보 경로(최대 k개)를 만든다."""
    risk = {e["id"]: edge_risk(e, nodes, threats) for e in edges}
    out, seen, pen = [], set(), {}
    for lam in LAMS + [0.0, 0.0, 0.3, 0.3]:
        adj = build_adj(nodes, edges, lambda e: (edge_minutes(e, kind) + lam * risk[e["id"]] * 100) * pen.get(e["id"], 1.0)
                        if passable(e, gross) else None)
        _, path, es = dijkstra(adj, DEPOT, dest)
        for e in es:
            pen[e["id"]] = pen.get(e["id"], 1.0) * 1.3
        if path and tuple(path) not in seen and len(out) < k:
            seen.add(tuple(path))
            surv = float(np.prod([1 - risk[e["id"]] for e in es]))
            out.append({"path": path, "min": sum(edge_minutes(e, kind) for e in es), "km": sum(e["km"] for e in es),
                        "surv": surv, "risk_score": -np.log(surv) * 100, "threat_edges": sum(risk[e["id"]] > 0 for e in es)})
    return out


def utility(c, w):
    return c["min"] + w * c["risk_score"]          # 낮을수록 좋음: 소요시간(분) + 위험 가중치 x 위험 점수


# 1) [교육용가상자료] 과거 수송통제관 승인 사례 40건: 숨은 선호(w=1.2)와 판단 편차(잡음)로 한 경로를 승인
rng = np.random.default_rng(4)
cases = []
for i in range(60):
    th = make_threats(seed=100 + i, n=int(rng.integers(2, 5)))
    cand = candidates(str(rng.choice(UNITS)), th)
    if len(cand) < 2:
        continue
    approved = int(np.argmin([utility(c, 1.2) + rng.normal(0, 1.0) for c in cand]))
    cases.append({"cand": cand, "approved": approved})
train, test = cases[:len(cases) // 2], cases[len(cases) // 2:]


def agreement(cs, w, top=1):
    return np.mean([c["approved"] in np.argsort([utility(x, w) for x in c["cand"]])[:top] for c in cs])


def loglik(cs, w, T=1.5):
    """선택 확률 P(i) ~ exp(-효용/T) 가정의 로그우도(조건부 로짓). 일치도보다 매끄러워 가중치 추정이 안정적이다."""
    ll = 0.0
    for c in cs:
        u = -np.array([utility(x, w) for x in c["cand"]]) / T
        ll += u[c["approved"]] - (u.max() + np.log(np.exp(u - u.max()).sum()))
    return ll


grid = np.round(np.arange(0, 3.01, 0.1), 1)
w_best = grid[int(np.argmax([loglik(train, w) for w in grid]))]
print(f"과거 승인 사례 {len(cases)}건(학습 {len(train)} / 검증 {len(test)}), 학습된 위험 가중치 w = {w_best}")
rows = [{"모델": "Baseline(최단시간, w=0)", "Top-1 일치도": agreement(test, 0.0), "Top-3 포함률": agreement(test, 0.0, 3)},
        {"모델": f"위협 회피 추천(w={w_best})", "Top-1 일치도": agreement(test, w_best), "Top-3 포함률": agreement(test, w_best, 3)}]
res = pd.DataFrame(rows)
print(res.to_string(index=False, float_format=lambda v: f"{v:.1%}"))
acc = rows[1]["Top-1 일치도"]
print(f"추천 정확도 {acc:.1%} (목표 85% 이상: {'달성' if acc >= 0.85 else '미달'})")

# 2) 현재 요청에 대한 Top-3 추천과 추천 사유
threats = make_threats()
dest, cargo = "N0307", 38.0
cand = candidates(dest, threats)
order = np.argsort([utility(c, w_best) for c in cand])[:3]
best_u = utility(cand[order[0]], w_best)
print(f"\n[요청: {dest} 보급 {cargo}t] 추천 Top-3 (유사도 = 1순위 대비 효용 비율)")
base = min(cand, key=lambda c: c["min"])
for rank, i in enumerate(order, 1):
    c = cand[i]
    print(f" {rank}순위 소요 {c['min']:.0f}분, 거리 {c['km']:.1f}km, 위협 통과 {c['threat_edges']}구간, 생존확률 {c['surv']:.1%}, 유사도 {best_u / utility(c, w_best):.2f}")
top = cand[order[0]]
print(f" 추천 사유: 최단시간 경로 대비 소요 {top['min'] - base['min']:+.0f}분, 위협 통과 {base['threat_edges']}->{top['threat_edges']}구간, 생존확률 {base['surv']:.1%}->{top['surv']:.1%}")


# 3) 적재 효율: 차종 조합(트럭 5t, 트레일러 20t) 최적화 - 교량 하중 때문에 트레일러 경로가 다를 수 있음
def plan(cargo, allow_trailer=True):
    best = None
    for n_trl in range(0, 4 if allow_trailer else 1):
        n_trk = int(max(0, np.ceil((cargo - 20 * n_trl) / 5)))
        if n_trk > 10:
            continue
        cost = 0.0
        if n_trl:
            c = candidates(dest, threats, "trailer", 50)
            if not c:
                continue
            cost += n_trl * min(c, key=lambda x: utility(x, w_best))["km"] * ASSETS["trailer"]["cost_km"]
        cost += n_trk * top["km"] * ASSETS["truck"]["cost_km"]
        cap = 20 * n_trl + 5 * n_trk
        cand_plan = {"트레일러": n_trl, "트럭": n_trk, "수송능력(t)": cap, "적재 효율": cargo / cap, "비용(단가x km)": round(cost, 1)}
        if best is None or cost < best["비용(단가x km)"]:
            best = cand_plan
    return best


p0, p1 = plan(cargo, False), plan(cargo)
print("\n[적재 계획 비교]")
print(pd.DataFrame([{"방식": "트럭만 배차", **p0}, {"방식": "차종 조합 최적화", **p1}]).to_string(index=False, float_format=lambda v: f"{v:.1%}" if v < 1 else f"{v:.1f}"))
print(f"비용 {1 - p1['비용(단가x km)'] / p0['비용(단가x km)']:.1%} 절감, 투입 차량 {p0['트럭']}대 -> {p1['트럭'] + p1['트레일러']}대")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for e in edges:
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    ax[0].plot([x1, x2], [y1, y2], color="lightgray", lw=1)
for t in threats:
    ax[0].add_patch(plt.Circle((t["x"], t["y"]), t["r"], color="red", alpha=0.15))
for rank, (i, col) in enumerate(zip(order, ["tab:green", "tab:blue", "tab:orange"]), 1):
    xy = np.array([nodes[n] for n in cand[i]["path"]]); ax[0].plot(xy[:, 0], xy[:, 1], color=col, lw=3 - 0.6 * rank, label=f"rank {rank}")
xy = np.array([nodes[n] for n in base["path"]]); ax[0].plot(xy[:, 0], xy[:, 1], "k--", lw=1, label="shortest-time")
ax[0].set_aspect("equal"); ax[0].legend(fontsize=8); ax[0].set_title("Threat-aware route recommendation (synthetic)")
ax[1].plot(grid, [agreement(train, w) for w in grid], "o-", label="train")
ax[1].plot(grid, [agreement(test, w) for w in grid], "s-", label="test")
ax[1].axhline(0.85, ls="--", c="gray"); ax[1].set_xlabel("risk weight w"); ax[1].set_ylabel("Top-1 agreement"); ax[1].legend()
ax[1].set_title("Agreement with past approvals")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
과거 승인 사례 60건(학습 30 / 검증 30), 학습된 위험 가중치 w = 2.3
                 모델  Top-1 일치도  Top-3 포함률
Baseline(최단시간, w=0)      43.3%     100.0%
    위협 회피 추천(w=2.3)      90.0%      96.7%
추천 정확도 90.0% (목표 85% 이상: 달성)

[요청: N0307 보급 38.0t] 추천 Top-3 (유사도 = 1순위 대비 효용 비율)
 1순위 소요 60분, 거리 50.0km, 위협 통과 0구간, 생존확률 100.0%, 유사도 1.00
 2순위 소요 66분, 거리 50.9km, 위협 통과 0구간, 생존확률 100.0%, 유사도 0.91
 3순위 소요 73분, 거리 55.8km, 위협 통과 0구간, 생존확률 100.0%, 유사도 0.82
 추천 사유: 최단시간 경로 대비 소요 +1분, 위협 통과 3->0구간, 생존확률 57.8%->100.0%

[적재 계획 비교]
       방식  트레일러  트럭  수송능력(t)  적재 효율  비용(단가x km)
   트럭만 배차     0   8       40  95.0%       400.0
차종 조합 최적화     2   0       40  95.0%       287.1
비용 28.2% 절감, 투입 차량 8대 -> 2대
```
</details>
